# LinQ Safety: ночное продолжение, ориентир 4–5 часов

Пять задач на двух T4. **People не исключён:** продолжение 17 завершённых эпох,
ещё до 20 минут / максимум 25 эпох суммарно. Каски продолжаются с 3 завершённых
эпох; жилеты впервые стартуют с COCO pretrained. Детекторы YOLO12m — максимум
40 эпох, классификаторы ResNet50 — до 12. Сохраняется optimizer исходных детекторов,
но расписание LR пересчитывается под меньший общий предел эпох.

Данные полные, исходные train/val/test неизменны; исключены известные ошибочные crop.
Лучшие веса выбираются на validation, test оценивается отдельно. СИЗ не обучает
падения. Зоны — геометрические правила поверх треков. Качество >0.95 не гарантируется.

Запускать **Save & Run All** с GPU T4 x2 и Internet. Inputs: приватный расширенный
Dataset и output версии **361611151**. Без last.pt с optimizer запуск остановится.
Каждую эпоху: best, last и отдельный resume-last с optimizer; предыдущая копия
сохраняется атомарно. Попытка внешнего backup в приватный Kaggle Dataset проверяется
до обучения. При отсутствии прав загрузки это явно записывается: локальный checkpoint
защищает от ошибки процесса, но не гарантирует сохранность при потере всей VM.

In [ ]:
import time
NIGHT_START=time.monotonic()
DEADLINE=NIGHT_START+285*60
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet',
 'ultralytics==8.4.88','onnx==1.19.1','protobuf==5.29.5','onnxruntime==1.29.0',
 'pyyaml==6.0.3','kagglehub'],check=True)
from pathlib import Path
import json,shutil,hashlib,zipfile,torch
WORK=Path('/kaggle/working');WORK.mkdir(exist_ok=True)
assert torch.cuda.is_available() and torch.cuda.device_count()>=2,'Select GPU T4 x2 before committing.'
print('NIGHT_GPU',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
OUTPUT=WORK/'night-training-output';OUTPUT.mkdir(exist_ok=True)

In [ ]:
SCRIPTS=WORK/'scripts';SCRIPTS.mkdir(exist_ok=True)
(SCRIPTS/'train_ppe.py').write_text('"""Train one PPE task; test is evaluated only after validation selects weights."""\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nfrom training_checkpoint import atomic_copy,atomic_torch_save\n\n# Sampled train-only QC found a vertically mirrored border labelled as helmet\n# although that crop contains only chin/shoulder. Keep raw data immutable and\n# exclude all classifier crops from this image (no complete detector task).\nQC_EXCLUDED_IMAGE_SHA256={\'8ab6abebd308d47cead1c3b6403e4bc767c288766d4113dde197c3609373a0c3\'}\n\n\ndef valid_classifier_file(path):\n    p=Path(path)\n    return p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\'} and p.stem[:64] not in QC_EXCLUDED_IMAGE_SHA256\n\n\ndef train_detector(args):\n    from ultralytics import YOLO,settings\n    # Disable optional account integrations only in this process. Do not write\n    # the user\'s persistent Ultralytics settings on a local training smoke.\n    dict.update(settings,{key:False for key in [\'wandb\',\'mlflow\',\'comet\',\'clearml\',\'neptune\',\'raytune\',\'hub\'] if key in settings})\n    import yaml\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    config=yaml.safe_load((Path(args.data)/args.task/\'data.yaml\').read_text())\n    config[\'path\']=str((Path(args.data)/args.task).resolve())\n    data_file=output/(args.task+\'-data.yaml\');data_file.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    last=output/args.task/\'weights/last.pt\'\n    continuation=None\n    if args.continue_checkpoint:\n        import torch,shutil\n        source=Path(args.continue_checkpoint)\n        # A retry must prefer this experiment\'s optimizer-preserving checkpoint.\n        local_resume=last.with_name(\'resume-last.pt\')\n        if args.resume and local_resume.exists():source=local_resume\n        # These are our own saved Kaggle checkpoints, not arbitrary uploaded pickle files.\n        state=torch.load(source,map_location=\'cpu\',weights_only=False)\n        if state.get(\'epoch\',-1)<0 or not state.get(\'optimizer\'):\n            raise ValueError(\'Exact continuation requires an unfinished checkpoint with optimizer state\')\n        if args.freeze:raise ValueError(\'Changing frozen parameter groups is incompatible with exact optimizer continuation\')\n        continuation={\'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\n                      \'previous_completed_epochs\':int(state[\'epoch\'])+1,\'optimizer_restored\':True}\n        groups=state[\'optimizer\'][\'param_groups\']\n        optimizer_name=\'MuSGD\' if any(g.get(\'use_muon\') for g in groups) else (\'AdamW\' if \'betas\' in groups[0] else \'SGD\')\n        # auto chooses a different optimizer when a shorter schedule drops below\n        # 10k iterations. Pin the saved optimizer family before restoring groups.\n        state[\'train_args\'].update(project=str(output.resolve()),name=args.task,data=str(data_file.resolve()),\n            save_dir=str((output/args.task).resolve()),epochs=args.epochs,optimizer=optimizer_name)\n        continuation[\'optimizer_family\']=optimizer_name\n        last.parent.mkdir(parents=True,exist_ok=True)\n        atomic_torch_save(state,last)\n        old_best=source.parent/\'best.pt\'\n        if old_best.exists() and old_best.resolve()!=(last.parent/\'best.pt\').resolve():\n            atomic_copy(old_best,last.parent/\'best.pt\')\n        history=source.parent.parent/\'results.csv\'\n        if history.exists():\n            rows=history.read_text().splitlines()[:int(state[\'epoch\'])+2]\n            (output/args.task/\'results.csv\').write_text(\'\\n\'.join(rows)+\'\\n\')\n        del state\n        args.resume=True\n    model=YOLO(str(last) if args.resume and last.exists() else args.weights)\n    resume=bool(args.resume and last.exists())\n    completed=resume and (model.ckpt.get(\'epoch\',0)<0 or model.ckpt.get(\'epoch\',0)+1>=args.epochs)\n    if completed and model.ckpt.get(\'epoch\',0)<0 and model.ckpt.get(\'train_args\',{}).get(\'epochs\')!=args.epochs:\n        raise ValueError(\'A completed checkpoint cannot resume a different epoch budget; use a new output experiment\')\n    if not completed:\n        if args.train_minutes:\n            # Ultralytics\' time= scheduler computes an epoch budget from zero.\n            # A callback limits added training while retaining the saved epoch/optimizer.\n            clock={}\n            def begin(trainer):clock[\'deadline\']=time.monotonic()+args.train_minutes*60\n            def limit(trainer):\n                if args.train_minutes and time.monotonic()>=clock[\'deadline\']:trainer.stop=True\n            model.add_callback(\'on_train_start\',begin)\n            model.add_callback(\'on_train_batch_end\',limit)\n        if args.train_minutes:\n            def preserve_resume(trainer):\n                if trainer.last.exists():\n                    resume_target=trainer.last.with_name(\'resume-last.pt\')\n                    if resume_target.exists():atomic_copy(resume_target,resume_target.with_name(\'resume-previous.pt\'))\n                    atomic_copy(trainer.last,resume_target)\n            model.add_callback(\'on_model_save\',preserve_resume)\n        model.train(data=str(data_file),epochs=args.epochs,\n                    imgsz=args.imgsz,batch=args.batch,device=args.device,seed=42,\n                    project=args.output,name=args.task,exist_ok=True,patience=8 if args.freeze else 30,\n                    close_mosaic=min(3 if args.freeze else 10,args.epochs),workers=2,amp=True,\n                    save_period=-1,resume=resume,freeze=args.freeze or None,\n                    cache=\'ram\' if args.cache_ram else False,\n                    time=None)\n    best=Path(args.output)/args.task/\'weights/best.pt\'\n    model=YOLO(str(best))\n    metrics=model.val(data=str(data_file),split=\'test\',\n                      imgsz=args.imgsz,batch=args.batch,device=args.device,\n                      project=str(output),name=args.task+\'-test\',exist_ok=True)\n    exported=Path(model.export(format=\'onnx\',imgsz=args.imgsz,batch=1,dynamic=False,\n                               simplify=False,opset=12,nms=False,device=\'cpu\'))\n    deployment={\'people\':\'person-yolo12m.onnx\',\'head_ppe\':\'ppe-head.onnx\',\'vest_ppe\':\'ppe-vest.onnx\'}[args.task]\n    import shutil\n    target=Path(args.output)/deployment;shutil.copyfile(exported,target)\n    if args.task==\'people\':shutil.copyfile(best,Path(args.output)/\'person-yolo12m.pt\')\n    result={\'task\':args.task,\'test_metrics\':{k:float(v) for k,v in metrics.results_dict.items()},\n            \'onnx_sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\n            \'linq_status\':\'not_compiled\',\'event_metrics\':\'not_evaluated\',\n            \'freeze_layers\':args.freeze,\'training_time_budget_minutes\':args.train_minutes,\n            \'continuation\':continuation,\n            \'actual_epochs\':sum(1 for _ in (output/args.task/\'results.csv\').open())-1}\n    (Path(args.output)/(args.task+\'-metrics.json\')).write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n\n\ndef train_classifier(args):\n    import numpy as np\n    import torch\n    from torch.utils.data import DataLoader\n    from torchvision import datasets,models,transforms\n    import onnx\n    torch.manual_seed(42)\n    root=Path(args.data)/(args.task+\'_classifier\')\n    mean=[.485,.456,.406];std=[.229,.224,.225]\n    common=[transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)]\n    train=datasets.ImageFolder(root/\'train\',transforms.Compose([\n        transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),\n        transforms.ColorJitter(.15,.15,.15,.03),*common[1:]]),is_valid_file=valid_classifier_file)\n    val=datasets.ImageFolder(root/\'val\',transforms.Compose(common),is_valid_file=valid_classifier_file)\n    test=datasets.ImageFolder(root/\'test\',transforms.Compose(common),is_valid_file=valid_classifier_file)\n    if train.class_to_idx!=val.class_to_idx or train.class_to_idx!=test.class_to_idx:\n        raise ValueError(\'Every split must have the same labelled classes\')\n    if min(len(train),len(val),len(test))<20:raise ValueError(\'Not enough reviewed crops for independent splits\')\n    device=torch.device(\'cuda:\'+args.device if str(args.device).isdigit() else args.device)\n    if args.classifier==\'resnet152\':\n        model=models.resnet152(weights=models.ResNet152_Weights.DEFAULT)\n    else:model=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)\n    model.fc=torch.nn.Linear(model.fc.in_features,len(train.classes));model.to(device)\n    if args.classifier_freeze_early:\n        for name,parameter in model.named_parameters():\n            parameter.requires_grad=name.startswith((\'layer4.\',\'fc.\'))\n    loaders={k:DataLoader(d,batch_size=args.batch,shuffle=k==\'train\',num_workers=2)\n             for k,d in [(\'train\',train),(\'val\',val),(\'test\',test)]}\n    weights=np.bincount(train.targets,minlength=len(train.classes))\n    loss_fn=torch.nn.CrossEntropyLoss(weight=torch.tensor(1/np.maximum(weights,1),dtype=torch.float32,device=device))\n    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-4)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    folder=Path(args.output);folder.mkdir(parents=True,exist_ok=True)\n    best_loss=float(\'inf\');checkpoint=folder/(args.task+\'-classifier.pt\')\n    start_epoch=0;last=folder/(args.task+\'-classifier-last.pt\')\n    if args.resume and last.exists():\n        state=torch.load(last,map_location=device,weights_only=True)\n        if state[\'classes\']!=train.classes:raise ValueError(\'Resume class order differs\')\n        if state.get(\'architecture\',args.classifier)!=args.classifier:raise ValueError(\'Resume classifier architecture differs\')\n        if state.get(\'freeze_early\',args.classifier_freeze_early)!=args.classifier_freeze_early:raise ValueError(\'Resume optimizer parameter groups differ\')\n        model.load_state_dict(state[\'state_dict\']);optimizer.load_state_dict(state[\'optimizer\'])\n        if state.get(\'scaler\'):scaler.load_state_dict(state[\'scaler\'])\n        if state.get(\'torch_rng\') is not None:torch.set_rng_state(state[\'torch_rng\'].cpu())\n        if state.get(\'cuda_rng\') and device.type==\'cuda\':torch.cuda.set_rng_state_all([v.cpu() for v in state[\'cuda_rng\']])\n        start_epoch=state[\'epoch\'];best_loss=state[\'best_loss\']\n    def evaluate(loader):\n        model.eval();losses=[];scores=[];targets=[]\n        with torch.no_grad():\n            for x,y in loader:\n                x,y=x.to(device),y.to(device);logits=model(x)\n                losses.append(float(loss_fn(logits,y)));scores.extend(logits.softmax(1).cpu().numpy());targets.extend(y.cpu().numpy())\n        return float(np.mean(losses)),np.asarray(scores),np.asarray(targets)\n    deadline=time.monotonic()+args.train_minutes*60 if args.train_minutes else float(\'inf\')\n    actual_epochs=start_epoch\n    for epoch in range(start_epoch,args.epochs):\n        model.train()\n        if args.classifier_freeze_early:\n            for name,module in model.named_children():\n                if name not in {\'layer4\',\'fc\'}:module.eval()\n        for x,y in loaders[\'train\']:\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=device.type==\'cuda\'):\n                loss=loss_fn(model(x.to(device)),y.to(device))\n            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n            if time.monotonic()>=deadline:break\n        validation_loss,_,_=evaluate(loaders[\'val\'])\n        print(json.dumps({\'epoch\':epoch+1,\'val_loss\':validation_loss}),flush=True)\n        if validation_loss<best_loss:\n            best_loss=validation_loss;atomic_torch_save({\'state_dict\':model.state_dict(),\'classes\':train.classes},checkpoint)\n        atomic_torch_save({\'state_dict\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n                    \'epoch\':epoch+1,\'best_loss\':best_loss,\'classes\':train.classes,\n                    \'architecture\':args.classifier,\'freeze_early\':args.classifier_freeze_early,\n                    \'torch_rng\':torch.get_rng_state(),\n                    \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []},last,previous=True)\n        actual_epochs=epoch+1\n        if time.monotonic()>=deadline:\n            print(\'Training time budget reached; selecting validated checkpoint and exporting\',flush=True)\n            break\n    saved=torch.load(checkpoint,map_location=device,weights_only=True);model.load_state_dict(saved[\'state_dict\'])\n    _,probabilities,truth=evaluate(loaders[\'val\'])\n    choices=[]\n    for threshold in np.linspace(.5,.99,50):\n        selected=probabilities.max(1)>=threshold;pred=probabilities.argmax(1)\n        per_class=[]\n        for label in range(len(train.classes)):\n            positives=selected & (pred==label);count=int(positives.sum())\n            per_class.append((count,int(((truth==label)&positives).sum())/max(count,1)))\n        if all(count>=20 and precision>=.95 for count,precision in per_class):\n            choices.append((float(selected.mean()),float(threshold)))\n    threshold=max(choices)[1] if choices else .8\n    _,test_scores,test_truth=evaluate(loaders[\'test\']);test_pred=test_scores.argmax(1)\n    # Rejections count as misses for recall on assessable test crops.\n    selected=test_scores.max(1)>=threshold;stats={}\n    for label,name in enumerate(train.classes):\n        tp=int(((test_pred==label)&(test_truth==label)&selected).sum())\n        fp=int(((test_pred==label)&(test_truth!=label)&selected).sum())\n        fn=int(((test_truth==label)&((test_pred!=label)|~selected)).sum())\n        precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)\n        stats[name]={\'precision\':precision,\'recall\':recall,\'f1\':2*precision*recall/max(precision+recall,1e-12),\'tp\':tp,\'fp\':fp,\'fn\':fn}\n    name=\'ppe-head-classifier.onnx\' if args.task==\'head_ppe\' else \'ppe-torso-classifier.onnx\'\n    target=folder/name;model.eval().cpu()\n    torch.onnx.export(model,torch.zeros(1,3,224,224),str(target),input_names=[\'images\'],\n                      output_names=[\'logits\'],opset_version=12,dynamo=False)\n    graph=onnx.load(target)\n    for key,value in {\'class_names\':json.dumps(train.classes),\'rgb_mean\':json.dumps(mean),\'rgb_std\':json.dumps(std),\n                      \'abstain_threshold\':str(threshold),\'crop_policy\':\'proposal_context_1.3\',\n                      \'resize_algorithm\':\'pil_bilinear\'}.items():\n        item=graph.metadata_props.add();item.key=key;item.value=value\n    onnx.save(graph,target)\n    report={\'classes\':train.classes,\'threshold\':threshold,\'validation_precision_target_met\':bool(choices),\n            \'actual_epochs\':actual_epochs,\'training_time_budget_minutes\':args.train_minutes,\n            \'frozen_early_layers\':args.classifier_freeze_early,\n            \'test\':stats,\'test_coverage\':float(selected.mean()),\'linq_status\':\'not_compiled\',\n            \'qc_excluded_image_sha256\':sorted(QC_EXCLUDED_IMAGE_SHA256),\n            \'split_crops\':{\'train\':len(train),\'val\':len(val),\'test\':len(test)},\n            \'warning\':\'Crop classification metrics are not event metrics; unobservable crops need a separate evaluation set.\'}\n    (folder/(args.task+\'-classifier-metrics.json\')).write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data/datasets/ppe-prepared\')\n    p.add_argument(\'--task\',choices=[\'people\',\'head_ppe\',\'vest_ppe\'],required=True)\n    p.add_argument(\'--mode\',choices=[\'detect\',\'classify\'],default=\'detect\')\n    p.add_argument(\'--weights\',default=\'yolo12m.pt\');p.add_argument(\'--classifier\',choices=[\'resnet50\',\'resnet152\'],default=\'resnet50\')\n    p.add_argument(\'--epochs\',type=int,default=150);p.add_argument(\'--imgsz\',type=int,default=640)\n    p.add_argument(\'--batch\',type=int,default=4);p.add_argument(\'--device\',default=\'0\')\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--freeze\',type=int,default=0)\n    p.add_argument(\'--cache-ram\',action=\'store_true\')\n    p.add_argument(\'--classifier-freeze-early\',action=\'store_true\')\n    p.add_argument(\'--train-minutes\',type=float,default=0)\n    p.add_argument(\'--continue-checkpoint\',help=\'Continue our saved detector optimizer/epoch state with an added time budget\')\n    p.add_argument(\'--output\',default=\'training-output\');a=p.parse_args()\n    if a.mode==\'classify\' and a.task==\'people\':p.error(\'Classifier is only for head/torso PPE\')\n    (train_detector if a.mode==\'detect\' else train_classifier)(a)\n',encoding='utf-8')
(SCRIPTS/'run_ppe_training.py').write_text('"""Run independent tasks on up to two GPUs, keeping logs and resume state."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor,wait\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport threading\nimport gc\nimport time\n\n\ndef run(args):\n    bounded=args.fast or args.night\n    deadline=time.monotonic()+args.max_minutes*60 if bounded else float(\'inf\')\n    import torch\n    if not torch.cuda.is_available():raise RuntimeError(\'Training requires a CUDA GPU\')\n    devices=list(range(min(torch.cuda.device_count(),2)))\n    queues=[[(\'detect\',\'people\'),(\'detect\',\'vest_ppe\'),(\'classify\',\'head_ppe\')],\n            [(\'detect\',\'head_ppe\'),(\'classify\',\'vest_ppe\')]]\n    if args.fast:queues[0]=[(\'detect\',\'vest_ppe\'),(\'detect\',\'people\'),(\'classify\',\'head_ppe\')]\n    if args.night:queues[0]=[(\'detect\',\'people\'),(\'detect\',\'vest_ppe\'),(\'classify\',\'head_ppe\')]\n    if len(devices)==1:queues=[queues[0]+queues[1]]\n    if args.jobs:\n        requested=set(args.jobs)\n        queues=[[job for job in queue if \'-\'.join(job) in requested] for queue in queues]\n    checkpoint_weights={};lineage={}\n    if bounded and not args.checkpoint_root:\n        raise ValueError(\'Fast continuation requires --checkpoint-root; refusing to restart trained tasks from stock weights\')\n    if args.checkpoint_root:\n        from ultralytics import YOLO\n        for task,nc in [(\'people\',1),(\'head_ppe\',2)]:\n            checkpoint=Path(args.checkpoint_root)/task/\'weights/last.pt\'\n            if not checkpoint.is_file():raise FileNotFoundError(f\'Required saved checkpoint is missing: {checkpoint}\')\n            loaded=YOLO(str(checkpoint));state=loaded.ckpt\n            if state.get(\'epoch\',-1)<0 or not state.get(\'optimizer\'):\n                raise ValueError(f\'{task}: expected a partial checkpoint with optimizer state\')\n            if len(loaded.names)!=nc:raise ValueError(f\'{task}: checkpoint classes differ\')\n            lineage[task]={\'path\':str(checkpoint),\'sha256\':hashlib.sha256(checkpoint.read_bytes()).hexdigest(),\n                           \'completed_epochs\':int(state[\'epoch\'])+1,\n                           \'mode\':\'resume optimizer and epoch state; limit additional training time\'}\n            checkpoint_weights[task]=str(checkpoint)\n            print(\'RESTORED\',task,\'completed epochs\',lineage[task][\'completed_epochs\'],flush=True)\n            del loaded,state;gc.collect()\n    weights=None\n    if any(mode==\'detect\' for queue in queues for mode,_ in queue):\n        from ultralytics import YOLO\n        print(\'Preparing YOLO12m initialization once before parallel workers\',flush=True)\n        initial=YOLO(\'yolo12m.pt\');weights=str(Path(initial.ckpt_path).resolve())\n        del initial;gc.collect()\n        # Ultralytics\' AMP sanity check loads this tiny official checkpoint.\n        # Fetch it once so concurrent trainers cannot overwrite one download.\n        initial=YOLO(\'yolo26n.pt\');del initial;gc.collect()\n    if any(mode==\'classify\' for queue in queues for mode,_ in queue):\n        from torchvision import models\n        print(\'Preparing classifier initialization once before parallel workers\',flush=True)\n        enum=models.ResNet152_Weights if args.classifier==\'resnet152\' else models.ResNet50_Weights\n        initial=enum.DEFAULT.get_state_dict(progress=True,check_hash=True)\n        del initial;gc.collect()\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    if lineage:(output/\'checkpoint-lineage.json\').write_text(json.dumps(lineage,indent=2)+\'\\n\')\n    script=Path(__file__).with_name(\'train_ppe.py\')\n    manifest=Path(args.data)/\'manifest.jsonl\'\n    identity={\'data\':str(Path(args.data).resolve()),\'epochs\':args.epochs,\'classifier_epochs\':args.classifier_epochs,\n              \'classifier\':args.classifier,\'batch\':args.batch,\'classifier_batch\':args.classifier_batch,\'imgsz\':args.imgsz,\n              \'fast\':args.fast,\'night\':args.night,\'job_minutes\':args.job_minutes,\'max_minutes\':args.max_minutes,\n              \'checkpoint_lineage\':lineage,\n              \'trainer_sha256\':hashlib.sha256(script.read_bytes()).hexdigest(),\n              \'data_manifest_sha256\':hashlib.sha256(manifest.read_bytes()).hexdigest() if manifest.exists() else \'bounded-smoke-subset\'}\n    progress=output/\'training-progress.json\';lock=threading.Lock()\n    prior=json.loads(progress.read_text()) if progress.exists() else {}\n    state={\'identity\':identity,\'jobs\':prior.get(\'jobs\',{}) if prior.get(\'identity\')==identity else {},\n           \'gpus\':[torch.cuda.get_device_name(i) for i in devices]}\n    def save():\n        temp=progress.with_suffix(\'.tmp\');temp.write_text(json.dumps(state,indent=2)+\'\\n\');temp.replace(progress)\n    def queue(device,jobs):\n        for mode,task in jobs:\n            key=mode+\'-\'+task\n            if state[\'jobs\'].get(key,{}).get(\'status\')==\'done\':\n                print(\'SKIP completed\',key,flush=True);continue\n            if deadline-time.monotonic()<120:\n                with lock:\n                    state[\'jobs\'][key]={\'status\':\'skipped_deadline\',\'gpu\':device};save()\n                print(\'SKIPPED deadline\',key,flush=True);continue\n            log=output/(key+\'.log\')\n            budget=({\'detect-people\':20,\'detect-head_ppe\':175,\'detect-vest_ppe\':105,\n                     \'classify-head_ppe\':55,\'classify-vest_ppe\':40}.get(key,args.job_minutes) if args.night else args.job_minutes)\n            budget=min(budget,max(1,(deadline-time.monotonic())/60-15)) if bounded else budget\n            epochs=25 if args.night and task==\'people\' else args.epochs\n            cmd=[sys.executable,str(script),\'--data\',args.data,\'--output\',args.output,\'--task\',task,\n                 \'--mode\',mode,\'--device\',str(device),\'--epochs\',str(epochs if mode==\'detect\' else args.classifier_epochs),\n                 \'--batch\',str(args.batch if mode==\'detect\' else args.classifier_batch),\'--imgsz\',str(args.imgsz),\'--resume\']\n            if mode==\'classify\':cmd+=[\'--classifier\',args.classifier]\n            elif weights:cmd+=[\'--weights\',checkpoint_weights.get(task,weights)]\n            if bounded:\n                cmd+=[\'--train-minutes\',str(budget)]\n                if mode==\'classify\':cmd+=[\'--classifier-freeze-early\']\n                elif task in checkpoint_weights:cmd+=[\'--continue-checkpoint\',checkpoint_weights[task],\'--cache-ram\']\n                else:cmd+=[\'--cache-ram\']+([\'--freeze\',\'10\'] if args.fast else [])\n            with lock:\n                state[\'jobs\'][key]={\'status\':\'running\',\'gpu\':device,\'log\':log.name,\'training_budget_minutes\':budget};save()\n            print(\'START\',key,\'GPU\',device,flush=True)\n            env={**os.environ,\'PYTHONUNBUFFERED\':\'1\',\'OMP_NUM_THREADS\':\'2\',\'MKL_NUM_THREADS\':\'2\'}\n            with log.open(\'a\',encoding=\'utf-8\') as f:\n                try:\n                    result=subprocess.run(cmd,stdout=f,stderr=subprocess.STDOUT,env=env,\n                        timeout=min(deadline-time.monotonic(),(budget+25)*60) if bounded else None)\n                    returncode=result.returncode\n                except subprocess.TimeoutExpired:\n                    returncode=124\n                    print(\'Worker exceeded deadline; saved checkpoints are retained\',file=f,flush=True)\n            with lock:\n                state[\'jobs\'][key].update(status=\'done\' if returncode==0 else \'failed\',returncode=returncode);save()\n            print(\'DONE\' if returncode==0 else \'FAILED\',key,\'GPU\',device,flush=True)\n    with lock:save()\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\n        pending={pool.submit(queue,device,jobs) for device,jobs in zip(devices,queues)}\n        while pending:\n            done,pending=wait(pending,timeout=20)\n            for future in done:future.result()\n            with lock:running=[(key,dict(job)) for key,job in state[\'jobs\'].items() if job[\'status\']==\'running\']\n            for key,job in running:\n                try:\n                    with (output/job[\'log\']).open(\'rb\') as f:\n                        f.seek(0,2);f.seek(max(0,f.tell()-3000));tail=f.read().decode(\'utf-8\',errors=\'replace\')\n                    lines=re.sub(r\'\\x1b\\[[0-9;]*[A-Za-z]\',\'\',tail).replace(\'\\r\',\'\\n\').splitlines()\n                    print(key,\'GPU\',job[\'gpu\'],next((s.strip()[:220] for s in reversed(lines) if s.strip()),\'initializing\'),flush=True)\n                except OSError:pass\n    if any(job[\'status\']!=\'done\' for job in state[\'jobs\'].values()):\n        raise RuntimeError(\'One or more tasks failed; see training-progress.json and per-task logs\')\n    print(\'All requested tasks completed. Independent event quality remains unmeasured.\',flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--epochs\',type=int,default=150);p.add_argument(\'--classifier-epochs\',type=int,default=30)\n    p.add_argument(\'--imgsz\',type=int,default=640);p.add_argument(\'--batch\',type=int,default=4)\n    p.add_argument(\'--classifier-batch\',type=int,default=16)\n    p.add_argument(\'--classifier\',choices=[\'resnet50\',\'resnet152\'],default=\'resnet152\')\n    p.add_argument(\'--fast\',action=\'store_true\',help=\'Freeze pretrained early layers and limit each training stage\')\n    p.add_argument(\'--night\',action=\'store_true\',help=\'Five tasks: short person continuation, PPE detectors, ResNet50 classifiers\')\n    p.add_argument(\'--job-minutes\',type=float,default=15)\n    p.add_argument(\'--max-minutes\',type=float,default=75,help=\'Fast-mode overall worker deadline, including setup/export\')\n    p.add_argument(\'--checkpoint-root\',help=\'Saved training-output folder; required for fast continuation\')\n    p.add_argument(\'--jobs\',nargs=\'+\',choices=[\'detect-people\',\'detect-head_ppe\',\'detect-vest_ppe\',\'classify-head_ppe\',\'classify-vest_ppe\'])\n    run(p.parse_args())\n',encoding='utf-8')
(SCRIPTS/'training_checkpoint.py').write_text('"""Crash-safe replacement of our own training checkpoints on one filesystem."""\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef atomic_copy(source, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with Path(source).open(\'rb\') as incoming, temporary.open(\'wb\') as outgoing:\n        shutil.copyfileobj(incoming, outgoing)\n        outgoing.flush()\n        os.fsync(outgoing.fileno())\n    os.replace(temporary, target)\n\n\ndef atomic_torch_save(state, target, previous=False):\n    import torch\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    if previous and target.exists():\n        atomic_copy(target, target.with_name(target.stem + \'-previous.pt\'))\n    os.replace(temporary, target)\n',encoding='utf-8')
(SCRIPTS/'backup_training.py').write_text('"""Optional private Kaggle Dataset backup using the official authenticated SDK.\n\nNo credentials are read or printed by this script. A lack of write authentication\nis reported honestly; local epoch checkpoints remain available independently.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nfrom training_checkpoint import atomic_copy\n\n\ndef snapshot(root, destination):\n    files=[]\n    for source in sorted(root.rglob(\'*\')):\n        if not source.is_file():continue\n        if source.suffix not in {\'.pt\',\'.json\',\'.csv\',\'.yaml\',\'.log\'}:continue\n        if source.name.endswith((\'-previous.pt\',\'resume-previous.pt\')):continue\n        target=destination/source.relative_to(root)\n        atomic_copy(source,target)\n        files.append({\'path\':str(source.relative_to(root)),\n                      \'sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\'bytes\':target.stat().st_size})\n    (destination/\'backup-manifest.json\').write_text(json.dumps({\'created_utc\':time.time(),\'files\':files},indent=2))\n    return files\n\n\ndef upload_once(root, handle, probe=False):\n    with tempfile.TemporaryDirectory(prefix=\'linq-checkpoint-backup-\') as name:\n        destination=Path(name)\n        if probe:\n            (destination/\'backup-probe.json\').write_text(json.dumps({\'purpose\':\'private LinQ training checkpoint backup\',\'time\':time.time()}))\n        else:snapshot(root,destination)\n        # Native authentication may support downloads without write access.\n        # Do not call login() or silently request/create an API credential.\n        import kagglehub\n        kagglehub.dataset_upload(handle,str(destination),version_notes=\'LinQ checkpoint snapshot; optimizer and best weights\')\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--handle\',required=True)\n    p.add_argument(\'--once\',action=\'store_true\');p.add_argument(\'--probe\',action=\'store_true\')\n    p.add_argument(\'--interval\',type=int,default=1800)\n    args=p.parse_args();root=Path(args.root);root.mkdir(parents=True,exist_ok=True)\n    status=root/\'remote-backup-status.json\'\n    if args.once:\n        try:\n            upload_once(root,args.handle,args.probe)\n            status.write_text(json.dumps({\'verified_upload\':True,\'handle\':args.handle,\'time\':time.time(),\'probe\':args.probe}))\n            print(\'REMOTE_BACKUP_UPLOAD_CONFIRMED\',args.handle,flush=True)\n        except Exception as error:\n            status.write_text(json.dumps({\'verified_upload\':False,\'handle\':args.handle,\'time\':time.time(),\n                \'error_type\':type(error).__name__,\'warning\':\'Local files alone do not survive a destroyed VM.\'}))\n            print(\'REMOTE_BACKUP_UNAVAILABLE\',type(error).__name__,\'No credential or response body logged.\',flush=True)\n            raise SystemExit(2)\n        return\n    while True:\n        command=[sys.executable,__file__,\'--root\',str(root),\'--handle\',args.handle,\'--once\']\n        try:subprocess.run(command,timeout=300,check=False)\n        except subprocess.TimeoutExpired:print(\'REMOTE_BACKUP_TIMEOUT; training continues\',flush=True)\n        time.sleep(args.interval)\n\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')

In [ ]:
input_root=Path('/kaggle/input')
manifests=sorted(input_root.rglob('manifest.jsonl'))
assert len(manifests)==1,'Attach exactly one expanded PPE Dataset.'
FULL_DATA=manifests[0].parent
roots=[p.parent for p in input_root.rglob('training-progress.json')
       if (p.parent/'people/weights/last.pt').exists() and (p.parent/'head_ppe/weights/last.pt').exists()]
assert len(roots)==1,'Attach saved version 361611151 output; no fresh restart is allowed.'
CHECKPOINT_ROOT=roots[0]
from ultralytics import YOLO
lineage={}
for task,expected_epoch in [('people',17),('head_ppe',3)]:
    path=CHECKPOINT_ROOT/task/'weights/last.pt'
    loaded=YOLO(path);state=loaded.ckpt
    assert state['epoch']+1==expected_epoch and state.get('optimizer'),(task,'Unexpected checkpoint or missing optimizer')
    lineage[task]={'completed_epochs':state['epoch']+1,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
    print('RESTORED_INPUT',task,lineage[task],flush=True)
    del loaded,state
(OUTPUT/'initial-checkpoint-lineage.json').write_text(json.dumps(lineage,indent=2))
# Copy once onto the VM's fast local disk. Keep the 2.7GB corpus out of saved output.
DATA=Path('/tmp/linq-night-data')
if not (DATA/'.complete').exists():
    from concurrent.futures import ThreadPoolExecutor
    files=[p for p in FULL_DATA.rglob('*') if p.is_file()]
    def copy_one(source):
        target=DATA/source.relative_to(FULL_DATA);target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(source,target)
    with ThreadPoolExecutor(max_workers=8) as pool:
        for index,_ in enumerate(pool.map(copy_one,files)):
            if index%10000==0:print('LOCAL_DATA_COPY',index,'/',len(files),flush=True)
    (DATA/'.complete').write_text(hashlib.sha256((FULL_DATA/'manifest.jsonl').read_bytes()).hexdigest())
assert (DATA/'.complete').read_text()==hashlib.sha256((FULL_DATA/'manifest.jsonl').read_bytes()).hexdigest()
shutil.copyfile(DATA/'manifest.jsonl',OUTPUT/'data-manifest.jsonl')
print('DATA_READY full corpus, original splits; setup minutes',(time.monotonic()-NIGHT_START)/60,flush=True)

In [ ]:
BACKUP_HANDLE='yathebest/linq-night-checkpoints-20261024'
try:
    probe=subprocess.run([sys.executable,str(SCRIPTS/'backup_training.py'),'--root',str(OUTPUT),
     '--handle',BACKUP_HANDLE,'--once','--probe'],timeout=180)
except subprocess.TimeoutExpired:
    (OUTPUT/'remote-backup-status.json').write_text(json.dumps({'verified_upload':False,'error_type':'probe_timeout'}))
remote=json.loads((OUTPUT/'remote-backup-status.json').read_text()) if (OUTPUT/'remote-backup-status.json').exists() else {'verified_upload':False}
BACKUP_PROCESS=None
if remote.get('verified_upload'):
    BACKUP_PROCESS=subprocess.Popen([sys.executable,str(SCRIPTS/'backup_training.py'),'--root',str(OUTPUT),'--handle',BACKUP_HANDLE])
else:
    print('BACKUP_LIMITATION: remote upload not authenticated. Epoch backups and saved-version output only; VM-loss protection not verified.',flush=True)

In [ ]:
remaining=max(1,(DEADLINE-time.monotonic())/60-15)
print('NIGHT_PLAN people=20min; head=175min; vest=105min; ResNet50 head=55min vest=40min; parallel GPU queues',flush=True)
try:
    training=subprocess.run([sys.executable,str(SCRIPTS/'run_ppe_training.py'),'--data',str(DATA),
      '--output',str(OUTPUT),'--night','--epochs','40','--classifier-epochs','12',
      '--batch','8','--classifier','resnet50','--classifier-batch','64',
      '--max-minutes',str(remaining),'--checkpoint-root',str(CHECKPOINT_ROOT)])
    print('NIGHT_TRAINING_EXIT',training.returncode,flush=True)
finally:
    if BACKUP_PROCESS:
        BACKUP_PROCESS.terminate()
        try:BACKUP_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:BACKUP_PROCESS.kill()

In [ ]:
# Representative calibration uses TRAIN images only, never held-out test data.
import numpy as np
from PIL import Image
for task,name in [('people','person-yolo12m'),('head_ppe','ppe-head'),('vest_ppe','ppe-vest')]:
    files=sorted((DATA/task/'images/train').glob('*.jpg'))
    rng=np.random.default_rng(42)
    samples=[]
    for index in rng.choice(len(files),min(64,len(files)),replace=False):
        image=Image.open(files[int(index)]).convert('RGB');w,h=image.size;r=min(640/w,640/h)
        image=image.resize((round(w*r),round(h*r)))
        canvas=Image.new('RGB',(640,640),(114,114,114));canvas.paste(image,((640-image.width)//2,(640-image.height)//2))
        samples.append(np.asarray(canvas,dtype=np.float32).transpose(2,0,1)/255)
    np.savez_compressed(OUTPUT/(name+'-calibration.npz'),images=np.stack(samples))
progress=json.loads((OUTPUT/'training-progress.json').read_text())
complete=len(progress['jobs'])==5 and all(j['status']=='done' for j in progress['jobs'].values())
manifest={'recipe':'night-5-tasks','complete':complete,'elapsed_minutes':(time.monotonic()-NIGHT_START)/60,
 'data_manifest_sha256':hashlib.sha256((DATA/'manifest.jsonl').read_bytes()).hexdigest(),
 'test_scope':'complete original held-out test splits','classifier':'resnet50','detector':'yolo12m',
 'remote_backup':json.loads((OUTPUT/'remote-backup-status.json').read_text()),
 'linq_hardware_verified':False,'event_metrics':'not_evaluated'}
(OUTPUT/'run-manifest.json').write_text(json.dumps(manifest,indent=2))
if remote.get('verified_upload'):
    try:subprocess.run([sys.executable,str(SCRIPTS/'backup_training.py'),'--root',str(OUTPUT),'--handle',BACKUP_HANDLE,'--once'],timeout=300)
    except subprocess.TimeoutExpired:print('Final remote backup timed out; local artifacts are retained.',flush=True)
archive=WORK/'linq-night-trained-models.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUTPUT))
print('DOWNLOAD',archive,'ALL_FIVE_DONE',complete,flush=True)
assert complete,'Partial run: checkpoints/logs are retained; inspect failed jobs before deployment.'